# Raw source-record preprocessing

This notebook cleans the raw ChEMBL-style source records, keeps valid IC50 assays, converts values to nM, and saves a preprocessed CSV for downstream modeling.


## 1. Install/check preprocessing dependencies


In [1]:
# Install dependency if needed in a fresh environment
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rdkit-pypi"])
    from rdkit import Chem
    from rdkit.Chem import Descriptors



## 2. Imports and dataset paths


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors

repo_root = Path.cwd().resolve()
if not (repo_root / "data" / "raw" / "source_records.csv").exists():
    repo_root = repo_root.parent

raw_path = repo_root / "data" / "raw" / "source_records.csv"
output_path = repo_root / "data" / "processed" / "preprocessed_source_records.csv"
print(f"Reading raw data from: {raw_path}")


Reading raw data from: /Users/m.saikarthikeya/Desktop/XGBioactiveNew/data/raw/source_records.csv


## 3. Convert assay values to nanomolar units


In [3]:
def convert_to_nm(value, unit, smiles):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan

    if pd.isna(value):
        return np.nan

    unit = str(unit).strip().lower()
    if unit in {"nm"}:
        return value
    if unit in {"um", "µm", "μm"}:
        return value * 1000
    if unit in {"mm"}:
        return value * 1_000_000
    if unit in {"pm"}:
        return value / 1000
    if unit in {"ug.ml-1", "ug/ml", "µg/ml", "μg/ml", "ug.mL-1", "µg.ml-1", "μg.ml-1"}:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return np.nan
        mw = Descriptors.MolWt(mol)
        if pd.notna(mw) and mw > 0:
            return (value * 1_000_000) / mw
        return np.nan
    if "um" in unit or "µm" in unit or "μm" in unit:
        return value * 1000
    if "m" == unit:
        return value * 1_000_000
    return np.nan


## 4. Load, validate, and preprocess source records


In [4]:
df = pd.read_csv(raw_path)
print(f"Raw rows: {len(df)}")
df = df.drop_duplicates().reset_index(drop=True)

df = df.loc[
    df["standard_type"].astype(str).str.strip().str.upper().eq("IC50")
    & df["original_smiles"].notna()
    & df["standard_value"].notna()
].copy()

if "validity_flag" in df.columns:
    explicit_false = df["validity_flag"].astype(str).str.strip().str.lower().eq("false")
    df = df.loc[~explicit_false].copy()

df["Standard Value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df["Standard Units"] = df["standard_units"].fillna("").astype(str).str.strip()
df["Smiles"] = df["original_smiles"].astype(str)
df["Molecule ChEMBL ID"] = df["source_compound_id"].astype(str)

df["IC50_nM"] = df.apply(
    lambda row: convert_to_nm(row["standard_value"], row["standard_units"], row["original_smiles"]),
    axis=1,
)
df = df.dropna(subset=["Molecule ChEMBL ID", "Smiles", "IC50_nM"]).copy()

df["Activity"] = np.where(df["IC50_nM"] < 10_000, "Active", "Inactive")
processed = df[[
    "Molecule ChEMBL ID",
    "Smiles",
    "standard_type",
    "Standard Value",
    "Standard Units",
    "Activity",
]].rename(columns={
    "standard_type": "Standard Type",
})
processed = processed.reset_index(drop=True)

print(f"Preprocessed rows: {len(processed)}")
print(processed.head())


Raw rows: 28689
Preprocessed rows: 25179
  Molecule ChEMBL ID                                             Smiles  \
0        CHEMBL68920  Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...   
1        CHEMBL68920  Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...   
2        CHEMBL68920  Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...   
3        CHEMBL69960  Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...   
4        CHEMBL69960  Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...   

  Standard Type  Standard Value Standard Units Activity  
0          IC50            41.0             nM   Active  
1          IC50           300.0             nM   Active  
2          IC50          7820.0             nM   Active  
3          IC50           170.0             nM   Active  
4          IC50            40.0             nM   Active  


## 5. Save the processed dataset and summarize results


In [5]:
processed.to_csv(output_path, index=False)
print(f"Saved preprocessed dataset to: {output_path}")
print(processed["Activity"].value_counts().to_string())


Saved preprocessed dataset to: /Users/m.saikarthikeya/Desktop/XGBioactiveNew/data/processed/preprocessed_source_records.csv
Activity
Active      21468
Inactive     3711
